In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Per-block stacked bar chart ---
ax = axes[0]
block_names_short = [bn.replace("block_", "").replace("_", "\n", 1) for bn in block_claims.keys()]
match_counts = []
approx_counts = []
mismatch_counts = []
for bclaims in block_claims.values():
    match_counts.append(sum(1 for c in bclaims if c["status"] == "MATCH"))
    approx_counts.append(sum(1 for c in bclaims if c["status"] == "APPROXIMATE"))
    mismatch_counts.append(sum(1 for c in bclaims if c["status"] == "MISMATCH"))

x = np.arange(len(block_names_short))
width = 0.6
bars_match = ax.bar(x, match_counts, width, label="MATCH", color="#4CAF50")
bars_approx = ax.bar(x, approx_counts, width, bottom=match_counts, label="APPROXIMATE", color="#FFC107")
bottom_mm = [m + a for m, a in zip(match_counts, approx_counts)]
bars_mm = ax.bar(x, mismatch_counts, width, bottom=bottom_mm, label="MISMATCH", color="#F44336")

ax.set_xlabel("Verification Block")
ax.set_ylabel("Number of Claims")
ax.set_title("Claim Verification by Block")
ax.set_xticks(x)
ax.set_xticklabels(block_names_short, fontsize=8)
ax.legend(loc="upper right", fontsize=8)
ax.set_ylim(0, max(m + a + mm for m, a, mm in zip(match_counts, approx_counts, mismatch_counts)) + 1)

# --- Plot 2: Expected vs Recomputed scatter ---
ax2 = axes[1]
expected_vals = []
recomputed_vals = []
statuses = []
for c in all_claims:
    ev = c["expected_value"]
    rv = c["recomputed_value"]
    if ev is not None and rv is not None:
        expected_vals.append(float(ev))
        recomputed_vals.append(float(rv))
        statuses.append(c["status"])

colors = {"MATCH": "#4CAF50", "APPROXIMATE": "#FFC107", "MISMATCH": "#F44336"}
for status in ["MATCH", "APPROXIMATE", "MISMATCH"]:
    ex = [e for e, s in zip(expected_vals, statuses) if s == status]
    re_ = [r for r, s in zip(recomputed_vals, statuses) if s == status]
    if ex:
        ax2.scatter(ex, re_, c=colors[status], label=status, alpha=0.7, edgecolors="k", linewidths=0.5, s=50)

# Perfect agreement line
all_vals = expected_vals + recomputed_vals
if all_vals:
    lo, hi = min(all_vals), max(all_vals)
    margin = (hi - lo) * 0.05 if hi != lo else 1.0
    ax2.plot([lo - margin, hi + margin], [lo - margin, hi + margin], "k--", alpha=0.3, label="Perfect agreement")

ax2.set_xlabel("Expected Value")
ax2.set_ylabel("Recomputed Value")
ax2.set_title("Expected vs Recomputed (all claims)")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig("claim_verification_results.png", dpi=100, bbox_inches="tight")
plt.show()
print(f"\nOverall consistency rate: {consistency_rate:.1%} ({n_match}/{n_total} claims match)")

## Visualization

Two views of claim verification results:
1. **Per-block stacked bar chart** showing MATCH/APPROXIMATE/MISMATCH counts
2. **Scatter plot** of expected vs recomputed values across all claims (log scale for wide value range)

In [ ]:
# ---------------------------------------------------------------------------
# Output formatting (from original format_output function)
# ---------------------------------------------------------------------------
n_total = len(all_claims)
n_match = sum(1 for c in all_claims if c["status"] == "MATCH")
n_approx = sum(1 for c in all_claims if c["status"] == "APPROXIMATE")
n_mismatch = sum(1 for c in all_claims if c["status"] == "MISMATCH")
consistency_rate = n_match / n_total if n_total > 0 else 0.0

blocks_clean = []
blocks_flagged = []
for block_name, bclaims in block_claims.items():
    if any(c["status"] == "MISMATCH" for c in bclaims):
        blocks_flagged.append(block_name)
    else:
        blocks_clean.append(block_name)

print("=" * 60)
print(f"TOTAL CLAIMS: {n_total}")
print(f"  MATCH:       {n_match} ({100*n_match/n_total:.1f}%)")
print(f"  APPROXIMATE: {n_approx} ({100*n_approx/n_total:.1f}%)")
print(f"  MISMATCH:    {n_mismatch} ({100*n_mismatch/n_total:.1f}%)")
print(f"  Consistency: {consistency_rate:.4f}")
print(f"\nClean blocks ({len(blocks_clean)}): {blocks_clean}")
print(f"Flagged blocks ({len(blocks_flagged)}): {blocks_flagged}")

# Log mismatches
if n_mismatch > 0:
    print("\nMISMATCHED CLAIMS:")
    for c in all_claims:
        if c["status"] == "MISMATCH":
            print(f"  {c['claim_id']}: expected={c['expected_value']}, "
                  f"recomputed={c['recomputed_value']}, notes={c['notes']}")

# Per-block summary table
print("\n" + "-" * 60)
print(f"{'Block':<30s} {'Match':>6s} {'Approx':>6s} {'Mismatch':>8s} {'Status':>8s}")
print("-" * 60)
for block_name, bclaims in block_claims.items():
    bm = sum(1 for c in bclaims if c["status"] == "MATCH")
    ba = sum(1 for c in bclaims if c["status"] == "APPROXIMATE")
    bmm = sum(1 for c in bclaims if c["status"] == "MISMATCH")
    status = "CLEAN" if bmm == 0 else "FLAGGED"
    print(f"{block_name:<30s} {bm:>6d} {ba:>6d} {bmm:>8d} {status:>8s}")
print("-" * 60)

## Summary Statistics

Aggregate claim verification results across all blocks, compute consistency rate, and identify clean vs flagged blocks.

In [ ]:
# ---------------------------------------------------------------------------
# Parse claims from each block and re-verify using make_claim logic
# ---------------------------------------------------------------------------
block_claims = {}
all_claims = []

for block_data in data["datasets"]:
    block_name = block_data["dataset"]
    claims = []
    print(f"\n=== {block_name} ===")

    for example in block_data["examples"]:
        inp = json.loads(example["input"])
        out = json.loads(example["output"])

        # Re-verify using our make_claim function
        reverified = make_claim(
            claim_id=inp["claim_id"],
            claim_text=inp["claim_text"],
            expected=out["expected_value"],
            recomputed=out["recomputed_value"],
            source_experiment=inp["source_experiment"],
            source_eval_reference=inp["source_eval_reference"],
            tolerance=out["tolerance"],
            tol_type=out["tolerance_type"],
            notes=out.get("notes", ""),
        )

        original_status = out["status"]
        reverified_status = reverified["status"]
        match_icon = "OK" if original_status == reverified_status else "DIFFER"

        print(f"  {inp['claim_id']}: expected={out['expected_value']}, "
              f"recomputed={out['recomputed_value']}, "
              f"original={original_status}, reverified={reverified_status} [{match_icon}]")

        claims.append(reverified)

    block_claims[block_name] = claims
    all_claims.extend(claims)

print(f"\nTotal claims processed: {len(all_claims)}")

## Parse and Re-Verify Claims

Parse each claim from the loaded data's JSON-encoded `input`/`output` fields and re-run the tolerance-based verification logic to confirm each claim's status.

In [ ]:
def cohens_d_independent(a: np.ndarray, b: np.ndarray) -> float:
    """Cohen's d for independent samples using pooled SD."""
    na, nb = len(a), len(b)
    ma, mb = np.mean(a), np.mean(b)
    sa, sb = np.var(a, ddof=1), np.var(b, ddof=1)
    sp = np.sqrt(((na - 1) * sa + (nb - 1) * sb) / (na + nb - 2))
    if sp == 0:
        return 0.0
    return float((ma - mb) / sp)


def cohens_d_paired(a: np.ndarray, b: np.ndarray) -> float:
    """Cohen's d for paired observations (dz = mean(diff)/sd(diff))."""
    diff = a - b
    sd = np.std(diff, ddof=1)
    return float(np.mean(diff) / sd) if sd > 0 else 0.0


def hedges_g(a: np.ndarray, b: np.ndarray) -> float:
    """Hedges' g (pooled SD, bias-corrected)."""
    na, nb = len(a), len(b)
    ma, mb = np.mean(a), np.mean(b)
    sa, sb = np.var(a, ddof=1), np.var(b, ddof=1)
    sp = np.sqrt(((na - 1) * sa + (nb - 1) * sb) / (na + nb - 2))
    if sp == 0:
        return 0.0
    d = (ma - mb) / sp
    df = na + nb - 2
    correction = 1 - 3 / (4 * df - 1)
    return float(d * correction)


def friedman_chi2(rank_matrix: np.ndarray) -> float:
    """Friedman chi-squared statistic. rank_matrix: (N_datasets, k_methods)."""
    N, k = rank_matrix.shape
    rank_means = np.mean(rank_matrix, axis=0)
    grand_mean = (k + 1) / 2.0
    ss = np.sum((rank_means - grand_mean) ** 2)
    return float(12 * N / (k * (k + 1)) * ss * k)


def nemenyi_cd(k: int, N: int, alpha: float = 0.05) -> float:
    """Nemenyi critical difference. q_alpha from Table for k groups."""
    q_table = {
        2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728,
        6: 2.850, 7: 2.949, 8: 3.031, 9: 3.102, 10: 3.164
    }
    q = q_table.get(k, 3.031)
    return float(q * math.sqrt(k * (k + 1) / (6.0 * N)))


def bootstrap_spearman_ci(x: np.ndarray, y: np.ndarray,
                          n_resamples: int = None,
                          seed: int = None) -> tuple:
    """Bootstrap 95% CI for Spearman rho."""
    if n_resamples is None:
        n_resamples = N_RESAMPLES_BOOTSTRAP
    if seed is None:
        seed = SEED
    rng = np.random.RandomState(seed)
    n = len(x)
    rhos = []
    for _ in range(n_resamples):
        idx = rng.choice(n, size=n, replace=True)
        rho, _ = stats.spearmanr(x[idx], y[idx])
        if not np.isnan(rho):
            rhos.append(rho)
    rhos = np.array(rhos)
    return float(np.percentile(rhos, 2.5)), float(np.percentile(rhos, 97.5))

print("Statistical helpers defined: cohens_d_independent, cohens_d_paired, hedges_g,")
print("  friedman_chi2, nemenyi_cd, bootstrap_spearman_ci")

## Statistical Helpers

Helper functions from the original evaluation script: Cohen's d, Hedges' g, Friedman chi-squared, Nemenyi critical difference, and bootstrap Spearman CI.

In [ ]:
def make_claim(claim_id: str, claim_text: str, expected, recomputed,
               source_experiment: str, source_eval_reference: str,
               tolerance: float, tol_type: str = "absolute",
               notes: str = "") -> dict:
    """Create a claim record and determine MATCH/APPROXIMATE/MISMATCH status."""
    tolerance = tolerance * TOLERANCE_MULTIPLIER
    if expected is None or recomputed is None:
        status = "MISMATCH"
        notes = notes or "One of expected/recomputed is None"
    elif tol_type == "absolute":
        diff = abs(float(expected) - float(recomputed))
        if diff <= tolerance:
            status = "MATCH"
        elif diff <= tolerance * 5:
            status = "APPROXIMATE"
        else:
            status = "MISMATCH"
    elif tol_type == "relative":
        if float(expected) == 0:
            status = "MATCH" if float(recomputed) == 0 else "MISMATCH"
        else:
            ratio = abs(float(recomputed) / float(expected))
            if 1.0 / (1 + tolerance) <= ratio <= (1 + tolerance):
                status = "MATCH"
            elif 1.0 / (1 + tolerance * 5) <= ratio <= (1 + tolerance * 5):
                status = "APPROXIMATE"
            else:
                status = "MISMATCH"
    elif tol_type == "order_of_magnitude":
        if float(expected) == 0 or float(recomputed) == 0:
            status = "MATCH" if (float(expected) == 0 and float(recomputed) == 0) else "MISMATCH"
        else:
            log_ratio = abs(math.log10(abs(float(recomputed))) - math.log10(abs(float(expected))))
            if log_ratio <= 0.5:
                status = "MATCH"
            elif log_ratio <= 1.0:
                status = "APPROXIMATE"
            else:
                status = "MISMATCH"
    else:
        status = "MISMATCH"
        notes = f"Unknown tolerance type: {tol_type}"

    return {
        "claim_id": claim_id,
        "claim_text": claim_text,
        "expected_value": expected,
        "recomputed_value": recomputed,
        "source_experiment": source_experiment,
        "source_eval_reference": source_eval_reference,
        "tolerance": tolerance,
        "tolerance_type": tol_type,
        "status": status,
        "notes": notes
    }

print("make_claim() defined")

## Claim Verification Logic

The core of the cross-validation pipeline: each claim has an `expected_value` and a `recomputed_value`, compared using tolerance-based matching (absolute, relative, or order-of-magnitude). Status is `MATCH`, `APPROXIMATE`, or `MISMATCH`.

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
# Number of bootstrap resamples for Spearman CI (original: 10000)
N_RESAMPLES_BOOTSTRAP = 1000  # original: 10000

# Tolerance multiplier for sensitivity analysis (1.0 = original tolerances)
TOLERANCE_MULTIPLIER = 1.0

# Random seed for reproducibility
SEED = 42

In [ ]:
data = load_data()
print(f"Loaded data with {len(data.get('datasets', []))} blocks")
print(f"Metadata: {json.dumps(data['metadata'], indent=2)}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-7dffcf-balance-guided-oblique-trees-signed-spec/main/evaluation_iter7_cross_validate/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
import json
import math
import os
import sys
from collections import defaultdict

import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# All packages used are pre-installed on Colab; install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0')

# Cross-Validate All Paper Claims: Master Fact Sheet

This notebook demonstrates the cross-validation evaluation pipeline that verifies
**193 quantitative claims** across 6 verification blocks (A-F) by recomputing
claims from raw experiment data.

**Blocks covered:**
- **A**: Main results table + Friedman/Nemenyi
- **B**: Interpretability arity + Wilcoxon
- **C**: Signed-vs-unsigned ablation + Hedges' g
- **D**: Timing/scalability
- **E**: Frustration meta-diagnostic (bootstrap CI)
- **F**: Synthetic module recovery

The demo loads pre-computed claim results, re-verifies each claim using the
tolerance-based matching logic, and visualizes consistency across blocks.